# 📘 Modèle Pyomo généré automatiquement

## 📦 Imports

In [ ]:
from pyomo.environ import *
from pyomo.opt import SolverFactory
import pandas as pd

## 🔹 Model

In [ ]:
from pyomo.environ import *

model = ConcreteModel()

## 🔹 Sets

In [ ]:
model.employes = Set(initialize=[1, 2, 3, 4, 5])
model.taches = Set(initialize=[1, 2, 3, 4, 5])
model.ARC = Set(dimen=2, initialize=[(i0,i1) for i0 in model.employes for i1 in model.taches])

## 🔹 Parameters

In [ ]:
model.C = Param(model.employes, model.taches, initialize={(1, 1): 39.0, (1, 2): 65.0, (1, 3): 69.0, (1, 4): 66.0, (1, 5): 57.0, (2, 1): 64.0, (2, 2): 84.0, (2, 3): 24.0, (2, 4): 92.0, (2, 5): 22.0, (3, 1): 49.0, (3, 2): 50.0, (3, 3): 61.0, (3, 4): 31.0, (3, 5): 45.0, (4, 1): 48.0, (4, 2): 45.0, (4, 3): 55.0, (4, 4): 23.0, (4, 5): 50.0, (5, 1): 59.0, (5, 2): 34.0, (5, 3): 30.0, (5, 4): 34.0, (5, 5): 18.0}, within=NonNegativeReals)

## 🔹 Variables

In [ ]:
model.X = Var(model.employes, model.taches, domain=NonNegativeReals)

## 🔹 Constraints

In [ ]:
model.c0 = Constraint(expr=model.X[5, 5] <= 0)
model.c1 = Constraint(expr=model.X[5, 2] >= 1)
model.c2 = Constraint(expr=model.X[2, 3] <= 0)
model.c3 = Constraint(expr=model.X[2, 5] >= 1)
model.c4 = Constraint(expr=model.X[5, 3] <= 0)
model.c5 = Constraint(expr=model.X[5, 2] >= 1)
model.c_for_0 = ConstraintList()
for t in model.taches:
    model.c_for_0.add(sum(model.X[e, t] for e in model.employes) == 1)
model.c_for_1 = ConstraintList()
for e in model.employes:
    for t in model.taches:
        model.c_for_1.add(model.X[e, t] <= 1)

## 🔹 Objective

In [ ]:
model.obj = Objective(expr=sum(model.C[model.e, model.t] * model.X[model.e, model.t] for model.e,model.t in model.ARC), sense=minimize)

## ⚙️ Résolution du modèle

In [ ]:
solver = SolverFactory('highs')
result = solver.solve(model, tee=True)

print('✅ Solver status:', result.solver.status)
print('✅ Termination condition:', result.solver.termination_condition)

## 🎯 Valeur de la fonction objective

In [ ]:
for obj in model.component_objects(Objective, active=True):
    print(f'Objectif: {obj.name}')
    print(f'Valeur optimale: {obj():.4f}')
    print(f'Sens: {"Minimisation" if obj.sense == minimize else "Maximisation"}')

## 📊 Valeurs optimales des variables

In [ ]:
# Extraction des résultats dans un DataFrame
results_data = []
for v in model.component_objects(Var, active=True):
    for index in v:
        results_data.append({
            'Variable': v.name,
            'Index': str(index) if index != None else '-',
            'Valeur': v[index].value
        })

df_results = pd.DataFrame(results_data)
# Filtrer les valeurs non-nulles pour plus de clarté
df_results = df_results[df_results['Valeur'].notna()]
df_results = df_results[df_results['Valeur'] != 0]
df_results.style.format({'Valeur': '{:.4f}'}).set_caption('Variables de décision optimales')